In [5]:
import pandas as pd
import boto3
from sqlalchemy import create_engine
from io import StringIO
import re
import os
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURATION SÉCURISÉE
# ==========================================
print("1. Chargement de la configuration...")
load_dotenv()

# Récupération des variables (avec gestion d'erreur sur le nom du mot de passe)
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID") or os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY") or os.getenv("AWS_SECRET_KEY")
BUCKET_NAME = os.getenv("BUCKET_NAME")

DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
# On cherche DB_PASS ou DB_PASSWORD pour être sûr
DB_PASS = os.getenv("DB_PASS") or os.getenv("DB_PASSWORD") 

# Paramètres fixes
REGION = "eu-west-3"
S3_FILENAME = "booking_final_gps.csv"
DB_PORT = "5432"

if not DB_PASS:
    print("Error: ERREUR : Mot de passe introuvable dans le .env")
    # Stop le script ici si pas de mot de passe
    raise ValueError("Vérifiez votre fichier .env")

# Connexion S3
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)
s3 = session.client('s3')

# ==========================================
# 2. EXTRACT (Depuis S3)
# ==========================================
print(f"\n2. Téléchargement de '{S3_FILENAME}' depuis S3...")

try:
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=S3_FILENAME)
    csv_content = obj['Body'].read().decode('utf-8')
    df = pd.read_csv(StringIO(csv_content))
    print(f"OK: Données brutes reçues : {len(df)} lignes.")
except Exception as e:
    print(f"Error: Erreur S3 : {e}")
    raise e

# ==========================================
# 3. TRANSFORM (Nettoyage)
# ==========================================
print("\n3. Nettoyage des données...")

# Fonction regex pour la note
def clean_score_v2(s):
    if pd.isna(s): return 0.0
    match = re.search(r'(\d+[.,]\d+)', str(s))
    if match:
        return float(match.group(1).replace(',', '.'))
    return 0.0

# Application des nettoyages
df['score'] = df['score'].apply(clean_score_v2) # On renomme score_clean en score directement
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Suppression des lignes sans GPS
df_clean = df.dropna(subset=['latitude', 'longitude']).copy()

# Sélection et renommage pour SQL (Conventionnel : minuscules, anglais)
df_final = df_clean[[
    'nom_hotel', 'ville', 'score', 'prix', 
    'latitude', 'longitude', 'url', 'description'
]].rename(columns={
    'nom_hotel': 'name',
    'ville': 'city',
    'score': 'rating',
    'prix': 'price',
    'url': 'booking_url'
})

print(f"OK: Transformation terminée. {len(df_final)} lignes prêtes à l'envoi.")

# ==========================================
# 4. LOAD (Vers RDS)
# ==========================================
print("\n4. Injection dans la base de données (RDS)...")

try:
    # URL de connexion avec SSL OBLIGATOIRE
    db_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
    engine = create_engine(db_url)
    
    # Envoi par paquets de 50 pour éviter le crash du Kernel
    df_final.to_sql('hotels', engine, if_exists='replace', index=False, chunksize=50)
    
    print("-" * 40)
    print("SUCCESS: SUCCÈS TOTAL ! Pipeline ETL terminé.")
    print(f"   Table 'hotels' créée sur {DB_HOST}")
    print("-" * 40)
    
except Exception as e:
    print(f"Error: Erreur RDS : {e}")

1. Chargement de la configuration...

2. Téléchargement de 'booking_final_gps.csv' depuis S3...
OK: Données brutes reçues : 699 lignes.

3. Nettoyage des données...
OK: Transformation terminée. 699 lignes prêtes à l'envoi.

4. Injection dans la base de données (RDS)...
----------------------------------------
SUCCESS: SUCCÈS TOTAL ! Pipeline ETL terminé.
   Table 'hotels' créée sur jehdabooking.cavcokeiclsx.us-east-1.rds.amazonaws.com
----------------------------------------


In [6]:
%pip install plotly
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. On recharge la config
load_dotenv()
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS") or os.getenv("DB_PASSWORD")

# 2. Connexion à la base
print(f"CONN: Connexion à la base sur {DB_HOST}...")
db_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}?sslmode=require"
engine = create_engine(db_url)

# 3. La Requête de la Victoire
try:
    # On lit directement la table 'hotels' qu'on vient de créer
    df_verif = pd.read_sql("SELECT * FROM hotels LIMIT 5", engine)
    
    print("\nINFO: CONTENU DE LA BASE DE DONNÉES AWS RDS :")
    print(df_verif.to_string())
    print("\nOK: Tout est en ordre capitaine !")
    
except Exception as e:
    print(f"Error: Erreur de lecture : {e}")

Note: you may need to restart the kernel to use updated packages.
CONN: Connexion à la base sur jehdabooking.cavcokeiclsx.us-east-1.rds.amazonaws.com...

INFO: CONTENU DE LA BASE DE DONNÉES AWS RDS :
                                          name               city  rating  price   latitude  longitude                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         booking_url                      